# Step 2 of RAG: Storing Chunks in ChromaDB**RAG** (Retrieval-Augmented Generation) pipeline recap:1. Document Loading & Chunking (Step 1)2. **Storing chunks in a Vector Database** ← *this notebook*3. Retrieval (searching the vector database for relevant chunks)4. Passing retrieved chunks to an LLM to generate the final answerThis notebook is self-contained — it recreates a sample document, chunks it (recap of Step 1),and then focuses fully on **storing those chunks in ChromaDB** the right way: with embeddings,useful metadata, and stable IDs so we can retrieve, update, or trace every chunk back to its source.### What you will learn1. Recap: create and chunk a document2. Why metadata (source, chunk index, etc.) matters when storing chunks3. Setting up a ChromaDB collection with a Hugging Face embedding function4. Storing chunks with meaningful IDs and metadata5. Verifying what got stored6. Querying stored chunks (a preview of Step 3: Retrieval)7. Adding a second document to show multi-document storage

### Setup — Install Required Libraries

In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

## 1. Recap — Create and Chunk a Document (Step 1)We regenerate a small sample document and split it into chunks using`RecursiveCharacterTextSplitter`, exactly as in the Step 1 notebook.

In [ ]:
import osos.makedirs("docs", exist_ok=True)article_text = """Artificial Intelligence: An OverviewArtificial Intelligence, or AI, is the field of computer science focused on building systemsthat can perform tasks which normally require human intelligence. These tasks includeunderstanding language, recognizing images, making decisions, and learning from experience.Machine Learning is a subset of AI where systems learn patterns from data instead of followinghardcoded rules. Instead of programming every decision by hand, we feed the system many examples,and it learns the underlying pattern on its own. This approach powers everything from spam filtersto recommendation systems.Deep Learning takes Machine Learning further by using neural networks with many layers. Theselayered networks can automatically discover complex patterns in large amounts of data, such asrecognizing faces in photos or understanding the meaning of a sentence.Natural Language Processing, or NLP, is the branch of AI that focuses specifically on humanlanguage. NLP powers chatbots, translation tools, and search engines. A key building block ofmodern NLP is the embedding, which converts words or sentences into numeric vectors that capturemeaning.Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with alanguage model. Instead of relying only on what a language model memorized during training, RAGfirst retrieves relevant, up-to-date information from a document collection, then passes thatinformation to the model so it can generate a more accurate, grounded answer.Vector Databases store embeddings and allow fast similarity search across millions of documents.They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendationengines used by companies around the world today."""with open("docs/ai_overview.txt", "w") as f:    f.write(article_text)print("Document created: docs/ai_overview.txt")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplittersplitter = RecursiveCharacterTextSplitter(    chunk_size=300,    chunk_overlap=50,    separators=["\n\n", "\n", ". ", " ", ""])chunks = splitter.split_text(article_text)print("Number of chunks created:", len(chunks))for i, chunk in enumerate(chunks):    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---")    print(chunk)

## 2. Why Metadata Matters When Storing ChunksWhen we store a chunk in a vector database, we should **never** store just the raw text. We shouldalso attach metadata so that later, when a chunk is retrieved, we know:| Metadata field | Why it matters ||---|---|| `source` | Which original file/document this chunk came from || `chunk_index` | The chunk's position/order within that document || `char_count` | Useful for debugging or filtering overly short/long chunks |This makes results traceable — a RAG answer can cite "this came from `ai_overview.txt`, chunk 3".

In [ ]:
chunk_metadatas = [    {        "source": "ai_overview.txt",        "chunk_index": i,        "char_count": len(chunk)    }    for i, chunk in enumerate(chunks)]chunk_ids = [f"ai_overview_chunk_{i}" for i in range(len(chunks))]print("Example metadata for chunk 0:", chunk_metadatas[0])print("Example id for chunk 0:", chunk_ids[0])

## 3. Set Up ChromaDB with a Hugging Face Embedding FunctionWe use the free `all-MiniLM-L6-v2` model so ChromaDB automatically converts every chunk's text intoan embedding vector as we add it.

In [ ]:
import chromadbfrom chromadb.utils import embedding_functionsclient = chromadb.Client()hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(    model_name="all-MiniLM-L6-v2")collection = client.create_collection(    name="rag_chunks",    embedding_function=hf_embedder)print("Collection created:", collection.name)

## 4. Store the Chunks in ChromaDBWe pass `documents` (the chunk text), `metadatas`, and `ids` all together. ChromaDB embeds eachchunk automatically using the embedding function we attached above.

In [ ]:
collection.add(    documents=chunks,    metadatas=chunk_metadatas,    ids=chunk_ids)print("Total chunks stored in ChromaDB:", collection.count())

## 5. Verify What Got StoredAlways double-check after a storage step — inspect a few chunks along with their metadata.

In [ ]:
stored = collection.get(    ids=chunk_ids,    include=["documents", "metadatas"])for cid, doc, meta in zip(stored["ids"], stored["documents"], stored["metadatas"]):    print(f"ID: {cid}")    print(f"Metadata: {meta}")    print(f"Text: {doc[:100]}...")    print("-" * 60)

## 6. Query the Stored Chunks (Preview of Step 3: Retrieval)Now that our chunks live in ChromaDB, we can search them by meaning. This is the foundation of thenext RAG step — retrieving the most relevant chunks for a user's question.

In [ ]:
results = collection.query(    query_texts=["How does RAG combine search with language models?"],    n_results=2,    include=["documents", "metadatas", "distances"])for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")    print(doc)    print("-" * 60)

**Discussion Point:** The `distance` score tells us how relevant each chunk is (lower = morerelevant). In a full RAG system, these top chunks would now be inserted into a prompt and sent to anLLM to generate a grounded final answer — that's Step 3 and Step 4.

## 7. Adding a Second Document (Multi-Document Storage)Real RAG systems store chunks from many documents in the same collection. The `source` metadata fieldis what lets us tell them apart later.

In [ ]:
second_doc_text = """Embeddings and Vector Search in PracticeModern search engines increasingly rely on embeddings rather than simple keyword matching.An embedding model converts a search query into the same vector space as the stored documents,allowing the system to find results that match the meaning of the query, not just its exact words.This is especially powerful for customer support systems, where a user might phrase a question verydifferently from how it appears in a help article, yet still expect to find the right answer."""with open("docs/embeddings_in_practice.txt", "w") as f:    f.write(second_doc_text)second_doc_chunks = splitter.split_text(second_doc_text)second_doc_metadatas = [    {        "source": "embeddings_in_practice.txt",        "chunk_index": i,        "char_count": len(chunk)    }    for i, chunk in enumerate(second_doc_chunks)]second_doc_ids = [f"embeddings_in_practice_chunk_{i}" for i in range(len(second_doc_chunks))]collection.add(    documents=second_doc_chunks,    metadatas=second_doc_metadatas,    ids=second_doc_ids)print("Chunks from second document added:", len(second_doc_chunks))print("Total chunks now stored in ChromaDB:", collection.count())

In [ ]:
# Query across BOTH documents at onceresults = collection.query(    query_texts=["how do search engines use embeddings"],    n_results=3,    include=["documents", "metadatas", "distances"])for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")    print(doc)    print("-" * 60)

**Discussion Point:** Notice the top result correctly comes from `embeddings_in_practice.txt` eventhough our collection also contains chunks from `ai_overview.txt`. This is exactly how a RAG systemsearches across an entire knowledge base to find the most relevant pieces, regardless of source.

## 8. Filtering Retrieval to a Single Source (Optional but Useful)Sometimes you only want to search within one document — combine semantic search with a metadatafilter using `where`.

In [ ]:
results = collection.query(    query_texts=["what is deep learning"],    n_results=2,    where={"source": "ai_overview.txt"},    include=["documents", "metadatas", "distances"])for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")    print(doc)    print("-" * 60)

## Key Takeaways (Recap for Students)| Concept | What We Did ||---|---|| Chunk storage | Stored each chunk as a `document` in ChromaDB, not the whole file at once || Embedding function | Attached `all-MiniLM-L6-v2` so ChromaDB embeds text automatically on `add()` || Metadata | Stored `source`, `chunk_index`, `char_count` with every chunk for traceability || IDs | Used predictable ids like `ai_overview_chunk_0` so chunks can be updated/deleted later || Multi-document storage | Added a second document into the same collection || Filtered retrieval | Used `where={"source": ...}` to search within a single document only |This collection of stored chunks is now ready for **Step 3: Retrieval**, where we take a user'squestion, search this same collection, and pass the top chunks to an LLM to generate the final answer.### Practice Exercises1. Add a third document of your own and confirm `collection.count()` reflects all chunks from all three.2. Re-chunk `ai_overview.txt` with `chunk_size=150` and re-store it under new ids (e.g. `..._v2_chunk_0`)   — compare retrieval results between the two chunk sizes.3. Write a helper function `add_document(collection, splitter, text, source_name)` that handles chunking,   metadata creation, and `collection.add()` in one reusable call.